In [1]:
import pandas as pd
import numpy as np

from scipy.stats import shapiro, levene, kruskal
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.multivariate.manova import MANOVA

import pingouin as pg

from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova

In [2]:
df = pd.read_csv("model_performance.csv")
df.head()

,model,max_length,train_time_sec,pred_time_sec,best_checkpoint,accuracy,f1,precision,recall
0,albert-base-v2,128,73.510133,2.010888,D:/huggingface_cache/classification_models/alb...,0.942247,0.941280,0.941824,0.942247
1,albert-base-v2,256,156.136250,4.518681,D:/huggingface_cache/classification_models/alb...,0.939873,0.938902,0.939336,0.939873
2,albert-base-v2,512,332.680437,9.699316,D:/huggingface_cache/classification_models/alb...,0.950158,0.949552,0.949776,0.950158
3,electra-small,128,35.851000,0.973974,D:/huggingface_cache/classification_models/ele...,0.950158,0.949265,0.950118,0.950158
4,electra-small,256,43.223512,1.017000,D:/huggingface_cache/classification_models/ele...,0.950158,0.950329,0.950572,0.950158


In [3]:
df = df[['model', 'max_length', 'train_time_sec', 'pred_time_sec',
         'accuracy', 'f1', 'precision', 'recall']].copy()

df['max_length'] = df['max_length'].astype(str)
df.head()

,model,max_length,train_time_sec,pred_time_sec,accuracy,f1,precision,recall
0,albert-base-v2,128,73.510133,2.010888,0.942247,0.941280,0.941824,0.942247
1,albert-base-v2,256,156.136250,4.518681,0.939873,0.938902,0.939336,0.939873
2,albert-base-v2,512,332.680437,9.699316,0.950158,0.949552,0.949776,0.950158
3,electra-small,128,35.851000,0.973974,0.950158,0.949265,0.950118,0.950158
4,electra-small,256,43.223512,1.017000,0.950158,0.950329,0.950572,0.950158


In [4]:
df.groupby('max_length')[['accuracy', 'f1', 'precision', 'recall']].agg(['mean', 'std', 'count'])

accuracy                        f1                 precision  \
                mean       std count      mean       std count      mean   
max_length                                                                 
128         0.893776  0.175369    15  0.883330  0.198866    15  0.912488   
256         0.948892  0.004707    15  0.948306  0.004929    15  0.948807   
512         0.951741  0.004435    15  0.951243  0.004806    15  0.951994   

                              recall                  
                 std count      mean       std count  
max_length                                            
128         0.112714    15  0.893776  0.175369    15  
256         0.004756    15  0.948892  0.004707    15  
512         0.004170    15  0.951741  0.004435    15

In [5]:
for g in df['max_length'].unique():
    stat, p = shapiro(df[df['max_length'] == g]['f1'])
    print(f"Shapiro test for chunk size {g}: statistic={stat:.4f}, p={p:.4f}")

groups = [df[df['max_length'] == g]['f1'] for g in df['max_length'].unique()]
stat, p = levene(*groups)
print(f"\nLevene test: statistic={stat:.4f}, p={p:.4f}")

Shapiro test for chunk size 128: statistic=0.4104, p=0.0000
Shapiro test for chunk size 256: statistic=0.8484, p=0.0165
Shapiro test for chunk size 512: statistic=0.8958, p=0.0822

Levene test: statistic=1.6445, p=0.2053


In [6]:
model = ols('f1 ~ C(max_length)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

,sum_sq,df,F,PR(>F)
C(max_length),0.044213,2.0,1.674941,0.199589
Residual,0.554334,42.0,NaN,NaN


In [7]:
tukey = pairwise_tukeyhsd(endog=df['f1'], groups=df['max_length'], alpha=0.05)
print(tukey)

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
   128    256    0.065  0.279 -0.0369 0.1669  False
   128    512   0.0679  0.249  -0.034 0.1698  False
   256    512   0.0029 0.9973  -0.099 0.1049  False
---------------------------------------------------


In [8]:
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis: statistic={stat:.4f}, p={p:.4f}")

Kruskal-Wallis: statistic=3.5367, p=0.1706


In [9]:
metrics = ['f1', 'precision', 'recall']

for col in metrics:
    stat, p = shapiro(df[col])
    print(f"Shapiro test for {col}: statistic={stat:.4f}, p={p:.4f}")

Shapiro test for f1: statistic=0.2221, p=0.0000
Shapiro test for precision: statistic=0.2419, p=0.0000
Shapiro test for recall: statistic=0.2151, p=0.0000


In [10]:
df[metrics].corr()

,f1,precision,recall
f1,1.000000,0.999646,0.998165
precision,0.999646,1.000000,0.997936
recall,0.998165,0.997936,1.000000


In [11]:
pg.box_m(data=df, dvs=metrics, group='max_length')

,Chi2,df,pval,equal_cov
box,337.531212,12.0,5.973768e-65,False


In [12]:
maov = MANOVA.from_formula('f1 + precision + recall ~ max_length', data=df)
print(maov.mv_test())

                     Multivariate linear model
                                                                    
--------------------------------------------------------------------
       Intercept          Value    Num DF  Den DF   F Value   Pr > F
--------------------------------------------------------------------
          Wilks' lambda     0.0000 3.0000 40.0000 286049.2759 0.0000
         Pillai's trace     1.0000 3.0000 40.0000 286049.2759 0.0000
 Hotelling-Lawley trace 21453.6957 3.0000 40.0000 286049.2759 0.0000
    Roy's greatest root 21453.6957 3.0000 40.0000 286049.2759 0.0000
--------------------------------------------------------------------
                                                                    
---------------------------------------------------------------------
           max_length        Value   Num DF   Den DF  F Value  Pr > F
---------------------------------------------------------------------
              Wilks' lambda  0.7871  6.0000  80.0000 

In [13]:
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova

metrics = ['f1', 'precision', 'recall']

# keep only needed columns and remove missing values
df_perma = df[['max_length'] + metrics].dropna().reset_index(drop=True)

# create matching string IDs
ids = df_perma.index.astype(str).tolist()

# build distance matrix
X = df_perma[metrics].values
dist_matrix = DistanceMatrix(squareform(pdist(X, metric='euclidean')), ids=ids)

# make dataframe index match the distance matrix IDs
df_perma.index = ids

# run PERMANOVA
permanova_result = permanova(dist_matrix, df_perma, column='max_length', permutations=999)
print(permanova_result)

method name               PERMANOVA
test statistic name        pseudo-F
sample size                      45
number of groups                  3
test statistic             1.636328
p-value                        0.11
number of permutations          999
Name: PERMANOVA results, dtype: object


In [14]:
from itertools import combinations
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova

metrics = ['f1', 'precision', 'recall']
results = []

for g1, g2 in combinations(df['max_length'].unique(), 2):
    sub = df[df['max_length'].isin([g1, g2])][['max_length'] + metrics].dropna().reset_index(drop=True)

    ids = sub.index.astype(str).tolist()
    X_sub = sub[metrics].values

    dist_sub = DistanceMatrix(
        squareform(pdist(X_sub, metric='euclidean')),
        ids=ids
    )

    sub.index = ids

    res = permanova(dist_sub, sub, column='max_length', permutations=999)

    results.append({
        'group1': g1,
        'group2': g2,
        'test_statistic': res['test statistic'],
        'p_value': res['p-value']
    })

pairwise_permanova = pd.DataFrame(results)
pairwise_permanova

,group1,group2,test_statistic,p_value
0,128,256,1.548955,0.346
1,128,512,1.721399,0.078
2,256,512,3.121992,0.087
